# Inspect bird images and labels

This notebook compares model-proposed boxes with reviewed boxes. It reads `data/` only and writes nothing.

Run it from the repository root kernel. Set `BIRD_DETECT_DATA_DIR` to inspect another data folder.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

from ml.check_labels import check_reviewed_labels, print_label_check
from ml.label_files import PROPOSED_LABELS, REVIEWED_LABELS, label_path_for, list_review_images, read_label_file
from ml.project_paths import DATA_DIR
from ml.summarize_boxes import summarize_reviewed_boxes

data_dir = Path(os.environ.get("BIRD_DETECT_DATA_DIR", DATA_DIR))
images = list_review_images(data_dir)
print(f"{len(images)} review images under {data_dir}")

## Label check and counts

In [ ]:
print_label_check(check_reviewed_labels(data_dir))
summarize_reviewed_boxes(data_dir)

## Proposed and reviewed boxes

Each row is one image. The left panel shows proposals. The right panel shows the reviewed label, or says it is pending.

In [ ]:
def draw_label_file(axis, image, label_path, color):
    if not label_path.exists():
        axis.set_title(f"{axis.get_title()} (none)")
        return
    width, height = image.size
    boxes, errors = read_label_file(label_path)
    for box in boxes:
        x0 = (box.center_x - box.width / 2) * width
        y0 = (box.center_y - box.height / 2) * height
        axis.add_patch(Rectangle((x0, y0), box.width * width, box.height * height, fill=False, edgecolor=color, linewidth=2))
    axis.set_title(f"{axis.get_title()} ({len(boxes)} boxes{', ' + str(len(errors)) + ' invalid' if errors else ''})")


def show_label_grid(image_paths, max_rows=12):
    rows = image_paths[:max_rows]
    if not rows:
        print("No images to show yet.")
        return
    figure, axes = plt.subplots(len(rows), 2, figsize=(12, 4 * len(rows)), squeeze=False)
    for (proposed_axis, reviewed_axis), image_path in zip(axes, rows):
        image = Image.open(image_path).convert("RGB")
        for axis, title in ((proposed_axis, "proposed"), (reviewed_axis, "reviewed")):
            axis.imshow(image)
            axis.set_axis_off()
            axis.set_title(f"{image_path.relative_to(data_dir)} {title}", fontsize=9)
        draw_label_file(proposed_axis, image, label_path_for(image_path, data_dir, PROPOSED_LABELS), "orange")
        draw_label_file(reviewed_axis, image, label_path_for(image_path, data_dir, REVIEWED_LABELS), "lime")
    figure.tight_layout()


show_label_grid(images)